There are $N$ children. Each child has $C$ boolean caracteristics such as _tall, thin, etc_.
The **difference** between two children is the number or characteristics on which they differ.
The children stand in a circle holding hands.
To maximise their enjoyment, the goal is to arrange them so as to **maximize the sum of the differences between adjacent children**.

In [1]:
from math import factorial
from random import Random

from circle_optimization import (
    Child, canonical_circle, circle_fitness, exact_best, inversion_mutation,
    pairwise_distances, random_individual, run_hill_climber,
    swap_mutation, unique_circles,
)

In [2]:
NUM_CHILDREN = 10
NUM_CARACTERISTICS = 5
EXACT_BENCHMARK_LIMIT = 10

The local `circle_optimization` library provides `Child` and the search functions. It represents a circle by child indices, so children with identical profiles remain distinct.

In [3]:
rng = Random()
CHILDREN = [
    Child(f'Child{index}', [bool(rng.getrandbits(1)) for _ in range(NUM_CARACTERISTICS)])
    for index in range(NUM_CHILDREN)
]
distances = pairwise_distances(CHILDREN, Child.difference)

In [4]:
for c in CHILDREN:
    print(c)

Child0:++-+-
Child1:+-+++
Child2:---++
Child3:-+-++
Child4:+---+
Child5:-+++-
Child6:--++-
Child7:+++++
Child8:+++++
Child9:+-+--


In [5]:
if len(CHILDREN) <= EXACT_BENCHMARK_LIMIT:
    num_unique_circles = sum(1 for _ in unique_circles(len(CHILDREN)))
else:
    num_unique_circles = factorial(len(CHILDREN) - 1) // 2
print('Unique circles:', num_unique_circles)

Unique circles: 181440


In [6]:
original_order = tuple(range(len(CHILDREN)))
print('Fitness of the original order:', circle_fitness(original_order, distances))

Fitness of the original order: 24


In [7]:
exact_circle = None
exact_score: int | None = None
if len(CHILDREN) <= EXACT_BENCHMARK_LIMIT:
    exact_circle, exact_score = exact_best(distances)
    print('Exact best score:', exact_score)
    print('Exact best circle:', ' -> '.join(CHILDREN[i].name for i in (*exact_circle, 0)))
else:
    print('Exact benchmark skipped for', len(CHILDREN), 'children')

Exact best score: 34
Exact best circle: Child0 -> Child1 -> Child3 -> Child9 -> Child2 -> Child5 -> Child4 -> Child7 -> Child6 -> Child8 -> Child0


In [8]:
print('Canonical orientation:', canonical_circle((0, *reversed(original_order[1:]))))

Canonical orientation: (0, 1, 2, 3, 4, 5, 6, 7, 8, 9)


In [9]:
solution = random_individual(len(CHILDREN), rng)
print('Random circle:', solution)

Random circle: (0, 1, 9, 8, 7, 4, 2, 5, 6, 3)


In [10]:
mutated = swap_mutation(solution, rng)
print('Swap mutation:', mutated)

Swap mutation: (0, 1, 5, 8, 7, 4, 2, 9, 6, 3)


In [11]:
inverted = inversion_mutation(solution, rng)
print('Inversion mutation (2-opt):', inverted)

Inversion mutation (2-opt): (0, 1, 9, 8, 6, 5, 2, 4, 7, 3)


In [12]:
print('Random start:', ' -> '.join(CHILDREN[i].name for i in (*solution, 0)))
print('Starting score:', circle_fitness(solution, distances))
print('One mutation:', ' -> '.join(CHILDREN[i].name for i in (*mutated, 0)))
print('Mutated score:', circle_fitness(mutated, distances))

Random start: Child0 -> Child1 -> Child9 -> Child8 -> Child7 -> Child4 -> Child2 -> Child5 -> Child6 -> Child3 -> Child0
Starting score: 22
One mutation: Child0 -> Child1 -> Child5 -> Child8 -> Child7 -> Child4 -> Child2 -> Child9 -> Child6 -> Child3 -> Child0
Mutated score: 24


### Random-mutation hill climber

Try a swap or segment reversal, accept only improvements, and restart from a new random circle when a run stalls. Score each proposal from the affected matrix edges instead of rescoring the entire circle; only accepted moves modify the circle.

The documented `run_hill_climber` implementation is shared with the GA notebook in [`circle_optimization/classic/hill_climbing.py`](circle_optimization/classic/hill_climbing.py). Its budget counts one score check per restart and per proposed move, including rejected or repeated moves. This differs from the GA's count of distinct complete-circle evaluations.

In [13]:
hill = run_hill_climber(
    distances, rng, evaluation_budget=20_000, max_restarts=20,
    attempts_per_restart=1000, stall_limit=200,
)
print('Hill-climb score:', hill['score'])
if exact_score is not None:
    print('Exact score:', exact_score)
    print('Gap to optimum:', exact_score - hill['score'])
else:
    print('Exact comparison unavailable above', EXACT_BENCHMARK_LIMIT, 'children')
print('Score checks (starts + moves):', hill['evaluations'])
print('Hill-climb circle:', ' -> '.join(CHILDREN[i].name for i in (*hill['circle'], 0)))

Hill-climb score: 34
Exact score: 34
Gap to optimum: 0
Score checks (starts + moves): 5603
Hill-climb circle: Child0 -> Child1 -> Child3 -> Child9 -> Child2 -> Child8 -> Child5 -> Child4 -> Child6 -> Child7 -> Child0
